# Recurrent 1.5-Bit Concept Bottleneck Model: Master Execution Notebook

This notebook runs the complete pipeline in sequence:
1. **Stage 1**: Supervised Fine-Tuning (PEFT/LoRA SFT) on Qwen 1.5B (distilled DeepSeek-R1) via Unsloth.
2. **Stage 2**: Batched Activation Hooking & Chunked Caching on Layer 14.
3. **Stage 3**: HybridCBM Representation Decomposition & Cosine Similarity Concept Translation.
4. **Stage 4**: T-TRM Recurrent Loop and CMR logic decider joint training under PST (Polynomial Surrogate Training) and monotonicity regularizations.

## 1. Setup Environment & Repository

Clone the code repository (if running in a fresh workspace) and navigate to the project root directory. This ensures all path references are correct.

In [ ]:
import os

# Check if we are already in the repository root
if not os.getcwd().endswith("recurrent-1.5bit-cbm"):
    if not os.path.exists("recurrent-1.5bit-cbm"):
        print("Cloning repository...")
        !git clone https://github.com/Borisz42/recurrent-1.5bit-cbm.git
    
    # Navigate into the repository directory
    %cd recurrent-1.5bit-cbm
    print("Pulling latest repository updates...")
    !git pull
else:
    print("Already in repository root. Pulling latest updates...")
    !git pull

## 2. Install Dependencies

Install the optimized libraries. On Kaggle, Unsloth must be installed using their specific wheels or git repository.

In [ ]:
# Install optimized deep learning and training libraries
!pip install -q lightning pytorch-lightning transformers accelerate safetensors datasets trl bitsandbytes>=0.46.1
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Stage 1: Base Model LoRA SFT

Fine-tune `DeepSeek-R1-Distill-Qwen-1.5B` in 4-bit precision on the instruction dataset. Formats instruction templates with reasoning cognitive traces using `<think>...</think>` tags.

In [ ]:
# Run SFT on the full dataset (runs for 1 epoch)
# Increased batch_size to 8 to utilize more GPU memory and speed up training
!python src/system1/train_sft.py \
    --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" \
    --dataset "tatsu-lab/alpaca" \
    --output_dir "./outputs" \
    --adapter_dir "./adapters" \
    --batch_size 8 \
    --gradient_accumulation_steps 2 \
    --epochs 1

## 4. Stage 2: Batched Layer Activation Caching

Hook Layer 14 of the fine-tuned model and save intermediate activations. To prevent CPU memory overflow on large datasets, the extractor automatically flushes batched tensors to chunked `.safetensors` files of 100 samples each.

In [ ]:
# Run activation extraction over the dataset in parallel on both GPUs
!python src/system1/extract_activations.py --rank 0 --world_size 2 --batch_size 16 --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" --adapter_dir "./adapters" --dataset "tatsu-lab/alpaca" --output_dir "./cached_activations" --layer_index 14 --chunk_size 100 & \
 python src/system1/extract_activations.py --rank 1 --world_size 2 --batch_size 16 --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" --adapter_dir "./adapters" --dataset "tatsu-lab/alpaca" --output_dir "./cached_activations" --layer_index 14 --chunk_size 100 & \
 wait


## 5. Stage 3: HybridCBM Optimization & Concept Translation

We load a representative subset of the cached activations (first 50 chunks) to train the Hybrid Concept Bottleneck Model. Limiting the chunk loading prevents memory overflow (OOM) on Kaggle's 16GB GPU / 30GB CPU RAM.

In [ ]:
import os
import torch
import torch.nn.functional as F
from safetensors.torch import load_file
from src.system1.hybrid_cbm import HybridCBM, DEFAULT_CONCEPTS
from transformers import CLIPTokenizer, CLIPTextModel

device = "cuda" if torch.cuda.is_available() else "cpu"
cache_dir = "./cached_activations"

# Limit to the first 50 chunk files (~5,000 samples) to prevent CPU RAM and VRAM OOM
chunk_files = sorted([os.path.join(cache_dir, f) for f in os.listdir(cache_dir) if f.endswith(".safetensors")])[:50]
print(f"Loading {len(chunk_files)} chunk files...")

activations_list = []
for f in chunk_files:
    chunk_data = load_file(f)
    activations_list.append(chunk_data["activations"])
    
# Keep activations on CPU to save VRAM
activations = torch.cat(activations_list, dim=0)
print(f"Loaded activations shape: {activations.shape} (stored on CPU)")

# Load CLIP text model to generate real static embeddings
print("Loading CLIP text model to generate real static embeddings...")
clip_model_name = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
text_model = CLIPTextModel.from_pretrained(clip_model_name).to(device)

def get_clip_embeddings(labels):
    inputs = tokenizer(labels, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
        embeddings = outputs.pooler_output
    return F.normalize(embeddings, p=2, dim=-1)

# Compute actual CLIP embeddings for default/static concepts
print("Computing CLIP embeddings for static concepts...")
static_embeddings = get_clip_embeddings(DEFAULT_CONCEPTS)

# Initialize HybridCBM with 5 dynamic concepts
n_dynamic = 5
hybrid_cbm = HybridCBM(
    n_dynamic=n_dynamic,
    clip_dim=512,
    emb_dim=activations.shape[-1],
    concepts=DEFAULT_CONCEPTS,
    clip_embeddings=static_embeddings
).to(device)

# Setup Optimizer with a stable learning rate (1e-3)
optimizer = torch.optim.Adam(hybrid_cbm.parameters(), lr=1e-3)

# Mini-batch parameters
batch_size = 4096
n_samples = activations.shape[0]
model_dtype = hybrid_cbm.proj_clip.weight.dtype

print("Training HybridCBM representation decomposition with LayerNorm & Activation De-correlation...")
for epoch in range(100):
    permutation = torch.randperm(n_samples)
    epoch_loss = 0.0
    epoch_ortho_w = 0.0
    epoch_ortho_a = 0.0
    
    for i in range(0, n_samples, batch_size):
        optimizer.zero_grad()
        
        # Get mini-batch, cast to model dtype, and send to GPU
        indices = permutation[i:i+batch_size]
        batch_x = activations[indices].to(device=device, dtype=model_dtype)
        
        # Forward pass on mini-batch
        z, x_rec, rec_loss = hybrid_cbm(batch_x)
        
        # Extract dynamic bottleneck activations: (batch_size, n_dynamic)
        z_dynamic = z[:, hybrid_cbm.n_static:]
        
        # 1. Weight Orthogonality Regularization (on decoder weights)
        dynamic_weights = hybrid_cbm.decoder.weight[:, hybrid_cbm.n_static:].T
        dynamic_weights_norm = F.normalize(dynamic_weights, p=2, dim=-1)
        weight_similarity = torch.matmul(dynamic_weights_norm, dynamic_weights_norm.T)
        identity = torch.eye(n_dynamic, device=device)
        weight_ortho_loss = torch.sum((weight_similarity - identity) ** 2)
        
        # 2. Activation Orthogonality Regularization
        z_dynamic_centered = z_dynamic - z_dynamic.mean(dim=0, keepdim=True)
        z_dynamic_norm = F.normalize(z_dynamic_centered, p=2, dim=0)
        activation_similarity = torch.matmul(z_dynamic_norm.T, z_dynamic_norm)
        activation_ortho_loss = torch.sum((activation_similarity - identity) ** 2)
        
        # Combined loss
        total_loss = rec_loss + 0.1 * weight_ortho_loss + 0.5 * activation_ortho_loss
        
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += rec_loss.item() * len(indices)
        epoch_ortho_w += weight_ortho_loss.item() * len(indices)
        epoch_ortho_a += activation_ortho_loss.item() * len(indices)
        
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / n_samples
        avg_ortho_w = epoch_ortho_w / n_samples
        avg_ortho_a = epoch_ortho_a / n_samples
        
        # Monitor the standard deviations on a validation slice
        with torch.no_grad():
            eval_x = activations[:1000].to(device=device, dtype=model_dtype)
            z_eval = torch.tanh(hybrid_cbm.proj_dynamic(hybrid_cbm.input_norm(eval_x)))
            stds = z_eval.std(dim=0).cpu().numpy()
            
        print(f"Epoch {epoch+1:03d} | Rec Loss: {avg_loss:.6f} | Weight Ortho: {avg_ortho_w:.6f} | Activation Ortho: {avg_ortho_a:.6f} | Stds: [{', '.join([f'{s:.4f}' for s in stds])}]")

# Save the trained HybridCBM checkpoint
torch.save(hybrid_cbm.state_dict(), "./hybrid_cbm.pt")
print("Saved HybridCBM checkpoint to ./hybrid_cbm.pt")

### Concept Translation

We project the learned dynamic concepts into the Candidate Concept Bank via CLIP space cosine similarity to assign human-understandable labels.

In [ ]:
# Define candidate concept bank or load from harvested JSON
import json
import os

json_path = "data/candidate_concepts.json"
if os.path.exists(json_path):
    print(f"Loading harvested candidate concepts from {json_path}...")
    with open(json_path, "r", encoding="utf-8") as f:
        candidate_labels = json.load(f)
else:
    print("Harvested JSON file not found. Initializing fallback candidate concept bank...")
    candidate_labels = [
        # Code Generation
        "code generation", "programming", "software development", "debugging", "regular expression",
        "string formatting", "variable declaration", "function definition", "syntax highlighting",
        "scripting", "variable assignment", "comment notation", "type casting", "loop iteration",
        "recursive call", "return output", "writing scripts", "bug tracking", "troubleshooting",
        "regex matching", "formatting code", "syntax formatting", "indentation syntax",
        "generating functions", "commenting code", "exception handling", "code refactoring",
        
        # Mathematical Reasoning
        "mathematical reasoning", "arithmetic calculation", "algebraic equation", "logical deduction",
        "binary comparison", "boolean logic", "matrix operation", "vector calculus", "probability theory",
        "counting", "addition", "subtraction", "multiplication", "division", "modulo", "exponentiation",
        "logic puzzles", "riddles logic", "calculating math", "multiplication division",
        "addition subtraction", "vector matrices", "counting elements", "discrete math",
        "statistical calculation", "modulo arithmetic", "recursive calculation",
        
        # Information Extraction
        "information extraction", "named entity recognition", "text parsing", "data structuring",
        "json parsing", "regex extraction", "database query", "schema mapping", "unstructured text",
        "entity linking", "ner extraction", "data parsing", "unstructured to structured",
        "json formatting", "database querying", "schema definition", "key value parsing",
        "xml parsing", "parsing tables", "metadata extraction",
        
        # Text Summarization
        "text summarization", "abstracting", "tldr generation", "bullet point summary", "paraphrasing",
        "text shortening", "executive summary", "synopsis generation", "condensation", "outline creation",
        "abstracting text", "summarizing documents", "extracting key points", "condensation text",
        "document outline",
        
        # Creative & Narrative Writing
        "creative narrative writing", "storytelling", "essay writing", "brainstorming ideas",
        "poetry generation", "dialogue writing", "character creation", "metaphor usage",
        "plot outline", "scriptwriting", "storytelling narrative", "fiction writing", "essay composing",
        "creative writing", "dialogue drafting", "metaphor creation", "character development",
        "creative brainstorming", "narrative descriptions",
        
        # Linguistic Translation
        "linguistic translation", "language conversion", "bilingual translation", "vocabulary lookup",
        "machine translation", "cross-lingual", "sentence translation", "idiomatic translation",
        "grammar correction", "dictionary definition", "translating languages", "vocabulary translation",
        "converting text language", "cross lingual translation", "converting text", "language conversion",
        "sentence translating", "grammar conversion",
        
        # Factual Recall & Q&A
        "factual recall qa", "question answering", "fact retrieval", "historical summary",
        "encyclopedia definition", "trivia answer", "knowledge retrieval", "information verification",
        "biographical detail", "concept explanation", "factual recall", "historical summaries",
        "dictionary definitions", "retrieving facts", "biographical recall", "trivia answering",
        "encyclopedic explanation", "verifying facts", "defining concepts",
        
        # Roleplay & Persona Simulation
        "roleplay persona simulation", "chatbot conversation", "stylistic constraint", "expert persona",
        "character dialogue", "conversational agent", "tone adjustment", "empathy simulation",
        "narrative roleplay", "fictional character", "roleplay conversation", "persona simulation",
        "chatbot agent", "stylistic constraints", "acting as expert", "acting as an expert",
        "tone simulation", "fictional roleplay", "expert advice simulation",
        
        # Linguistic Classification
        "linguistic classification", "sentiment analysis", "intent detection", "text categorization",
        "sorting list items", "topic classification", "spam filtering", "parts of speech tagging",
        "urgency detection", "language identification", "sorting lists", "text classification",
        "categorizing text", "spam detection", "parts of speech classification", "topic categorization",
        "urgency classification", "labeling sentences"
    ]

# Compute real normalized CLIP embeddings for candidate concepts
print("Computing real CLIP embeddings for candidate labels...")
candidate_embeddings = get_clip_embeddings(candidate_labels)

# Perform translation via Zero-Centered Pearson Correlation
with torch.no_grad():
    model_dtype = hybrid_cbm.proj_clip.weight.dtype
    
    # Use a representative subset of 10,000 samples to prevent GPU OOM
    eval_size = min(10000, activations.shape[0])
    activations_sub = activations[:eval_size].to(device=device, dtype=model_dtype)
    
    # 1. Project dataset activations to CLIP space
    x_clip = hybrid_cbm.proj_clip(hybrid_cbm.input_norm(activations_sub))  # Shape: (eval_size, clip_dim)
    x_clip_norm = F.normalize(x_clip, p=2, dim=-1)
    
    # 2. Get dynamic concept activations
    z_dynamic = torch.tanh(hybrid_cbm.proj_dynamic(hybrid_cbm.input_norm(activations_sub)))  # Shape: (eval_size, n_dynamic)
    
    # --- Zero-Center z_dynamic along the batch dimension (dim=0) ---
    z_dynamic_centered = z_dynamic - z_dynamic.mean(dim=0, keepdim=True)
    z_dynamic_norm = F.normalize(z_dynamic_centered, p=2, dim=0) 
    
    # 3. Compute raw activation scores for all candidate labels on the dataset
    candidate_embeds_norm = F.normalize(candidate_embeddings.to(dtype=model_dtype), p=2, dim=-1)
    candidate_activations = torch.matmul(x_clip_norm, candidate_embeds_norm.T)  # Shape: (eval_size, n_candidates)
    
    # --- Zero-Center candidate activations along the batch dimension (dim=0) ---
    candidate_activations_centered = candidate_activations - candidate_activations.mean(dim=0, keepdim=True)
    candidate_activations_norm = F.normalize(candidate_activations_centered, p=2, dim=0)
    
    # 4. Compute Pearson correlation matrix: (n_dynamic, n_candidates)
    correlation_matrix = torch.matmul(z_dynamic_norm.T, candidate_activations_norm)
    
    # Find the candidate with the highest absolute correlation
    abs_correlation = torch.abs(correlation_matrix)
    max_abs_corrs, max_indices = torch.max(abs_correlation, dim=-1)

print("Translated Dynamic Concepts (Smarter Pearson Translation):")
threshold = 0.35
for i in range(hybrid_cbm.n_dynamic):
    idx = max_indices[i].item()
    val = correlation_matrix[i, idx].item()
    
    if abs(val) < threshold:
        label = "unmapped (abstract activation pattern)"
    elif val < 0:
        label = f"not {candidate_labels[idx]}"
    else:
        label = candidate_labels[idx]
        
    print(f"  Dynamic Concept {i+1} -> Label: '{label}' (Correlation: {val:.4f})")

## 6. Stage 4: T-TRM Loop & Rule-Memory Optimization

We jointly optimize the `TTRMLoop` (PST logic gate parameters) and the `CMRModel` logic decider on the cached activations and target concepts using task classification losses and gate monotonicity regularizations.

In [ ]:
# Jointly train the T-TRM loop and CMR decider with speed and memory optimizations
!python -u src/t_trm/train_trm.py \
    --cache_dir "./cached_activations" \
    --hybrid_cbm_path "./hybrid_cbm.pt" \
    --epochs 50 \
    --batch_size 1024 \
    --num_workers 0 \
    --max_chunks 20


## 7. Stage 5: Formal Evaluation Matrix (CUE, Adversarial Steering, Rule Extraction)

We execute the formal testing suite to evaluate Concept Utilization Efficiency (CUE), Adversarial Steering noise tolerance, and print the extracted logical rules.

In [ ]:
# Run the formal evaluation suite with unbuffered printing
!python -u src/eval/run_tests.py \
    --cache_dir "./cached_activations" \
    --hybrid_cbm_path "./hybrid_cbm.pt" \
    --model_dir "./t_trm_outputs" \
    --max_chunks 20


## 8. Steered Inference & Verification on Custom Queries

We test the end-to-end neuro-symbolic pipeline on three custom general instruction queries. For each query, we output the System 1 causal generation alongside the System 2 Concept Activations and Decider Rules.

In [ ]:
import argparse
from src.eval.chatbot_app import SteeredChatbot

# Configure arguments to match Stage 4/5 configuration
args = argparse.Namespace(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit",
    adapter_dir="./adapters",
    hybrid_cbm_path="./hybrid_cbm.pt",
    model_dir="./t_trm_outputs",
    n_dynamic=5,
    clip_dim=512,
    n_latent=8,
    n_rules=10,
    layer_index=14,
    max_tokens=256,
    load_in_4bit=True,
    concepts_type="general",
    cli=True
)

# Instantiate the chatbot engine
print("Initializing SteeredChatbot...")
engine = SteeredChatbot(args)

# Define 3 specific general queries
general_queries = [
    "Explain the theory of relativity in simple terms for a child.",
    "Solve the mathematical equation: 3x + 5 = 20. Step by step.",
    "List 5 healthy breakfast options that are low in sugar."
]

# Run and log details for each query
for idx, query in enumerate(general_queries):
    print(f"\n=======================================================")
    print(f"QUERY {idx+1}: {query}")
    print(f"=======================================================")
    
    response, concepts, rules = engine.generate_and_reason(query)
    
    print("\n[System 1 Output (Generated Response)]")
    print(response.strip())
    
    print("\n[System 2 Debugger - Concept Activations]")
    for name, val in sorted(concepts.items(), key=lambda x: x[1], reverse=True)[:10]:
        bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
        print(f"  - {name:30s}: {bar} ({val*100:5.1f}%)")
        
    print("\n[System 2 Debugger - Active Logic Constraints]")
    print(rules)
    print("-------------------------------------------------------")